# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Simular la ingesta de datos ruidosos y procesarlos mediante un flujo ETL con capas RAW, QUARANTINE y CURATED.

In [1]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')

n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

1/3 Generando datos base perfectos en memoria...
2/3 Leyendo archivo desde disco e inyectando caos...
3/3 Dataset de ingesta listo. Total de filas: 5,050,000


## Fase 1: Profiling y Capa RAW
Perfilamos el archivo crudo y lo almacenamos en la capa inmutable RAW.

In [2]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

os.makedirs('./datalake/raw', exist_ok=True)
os.makedirs('./datalake/curated', exist_ok=True)
os.makedirs('./datalake/quarantine', exist_ok=True)

print("\n--- EXTRACT & LOAD (EL) ---")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet', engine='pyarrow', index=False)
print("¡Archivo RAW almacenado con éxito!")

--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---

1. Valores Nulos por Columna:
id_transaccion         0
id_cliente             0
monto             252510
categoria              0
estado                 0
dtype: int64

2. Registros Duplicados Exactos: 49,999

3. Resumen Estadístico de 'monto':
count    4,797,490.00
mean         1,463.00
std         45,882.62
min              5.00
25%            628.91
50%          1,252.56
75%          1,876.37
max      9,999,999.99
Name: monto, dtype: object

--- EXTRACT & LOAD (EL) ---
¡Archivo RAW almacenado con éxito!


## Fase 2: Transform (T) - Limpieza y Normalización
Se eliminan duplicados, se aplica descarte inteligente e imputación de nulos, se envían los outliers a QUARANTINE mediante IQR y se normalizan los datos restantes.

In [5]:
print("Iniciando Transformación...\n")

# 1. Eliminación de duplicados
antes = len(df_ingesta)
df_limpio = df_ingesta.drop_duplicates().copy()
print(f"Duplicados eliminados: {antes - len(df_limpio):,}")

# 2. Imputación robusta de Nulos (Mediana por categoría)
# RETO 2: Descarte inteligente
condicion_cancelado_nulo = (
    (df_limpio['estado'] == 'Cancelado') &
    (df_limpio['monto'].isnull())
)
filas_descartadas = condicion_cancelado_nulo.sum()
df_limpio = df_limpio.loc[~condicion_cancelado_nulo].copy()
print(f"Transacciones Canceladas eliminadas por monto nulo: {filas_descartadas:,}")

# Imputación con mediana por categoría para los demás estados
medianas = df_limpio.groupby('categoria')['monto'].transform('median')
df_limpio['monto'] = df_limpio['monto'].fillna(medianas)
print(f"Nulos restantes en monto: {df_limpio['monto'].isnull().sum()}")

# 3. Tratamiento de Outliers (Regla IQR)
Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

# RETO 1: Capa de Cuarentena
mascara_outliers = df_limpio['monto'] > limite_superior
df_cuarentena = df_limpio.loc[mascara_outliers].copy()
df_limpio = df_limpio.loc[~mascara_outliers].copy()

os.makedirs('./datalake/quarantine', exist_ok=True)
ruta_cuarentena = './datalake/quarantine/outliers_transacciones.parquet'
df_cuarentena.to_parquet(ruta_cuarentena, engine='pyarrow', index=False)
print(f"Outliers enviados a cuarentena: {len(df_cuarentena):,}")
print(f"Archivo guardado en: {ruta_cuarentena}")
print(f"Registros que continúan hacia Curated: {len(df_limpio):,}")

# 4. Normalización Min-Max (Escala de 0 a 1)
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()
if max_monto != min_monto:
    df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
else:
    df_limpio['monto_escalado'] = 0.0

print("Normalización Min-Max completada.")

# RETO 3: Estandarización Z-score
media_monto = df_limpio['monto'].mean()
std_monto = df_limpio['monto'].std()
df_limpio['monto_estandarizado'] = (df_limpio['monto'] - media_monto) / std_monto
print(f"Z-score completado. Media: {df_limpio['monto_estandarizado'].mean():.4f}, "
      f"Desv. estándar: {df_limpio['monto_estandarizado'].std():.4f}")

Iniciando Transformación...

Duplicados eliminados: 49,999
Transacciones Canceladas eliminadas por monto nulo: 62,513
Nulos restantes en monto: 0
Outliers enviados a cuarentena: 101
Archivo guardado en: ./datalake/quarantine/outliers_transacciones.parquet
Registros que continúan hacia Curated: 4,937,387
Normalización Min-Max completada.
Z-score completado. Media: -0.0000, Desv. estándar: 1.0000


## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el conjunto limpio en CURATED y verificamos el almacenamiento de las tres capas.

In [6]:
# 1. Guardar datos limpios en CURATED
ruta_curated = './datalake/curated/transacciones_model_ready.parquet'
df_limpio.to_parquet(ruta_curated, engine='pyarrow', index=False)
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 2. Estadística final
print("--- ESTADÍSTICA DEL DATO CURADO ---")
print(df_limpio[['monto', 'monto_escalado', 'monto_estandarizado']].describe().apply(
    lambda columna: columna.map(lambda x: f"{x:,.2f}")
))
display(df_limpio[['id_transaccion', 'categoria', 'estado', 'monto',
                   'monto_escalado', 'monto_estandarizado']].head(5))

print("\n--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'estado', 'monto', 'monto_escalado']].head(5))

# 3. Resumen de almacenamiento
ruta_raw = './datalake/raw/ingesta_transacciones_sucio.parquet'
ruta_quarantine = './datalake/quarantine/outliers_transacciones.parquet'

print("\n--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---")
print(f"RAW (Sucio): {os.path.getsize(ruta_raw)/(1024*1024):.2f} MB")
print(f"QUARANTINE (Outliers): {os.path.getsize(ruta_quarantine)/(1024*1024):.2f} MB")
print(f"CURATED (Limpio): {os.path.getsize(ruta_curated)/(1024*1024):.2f} MB")

print("\n--- CONTROL DE REGISTROS ---")
print(f"Registros en cuarentena: {len(df_cuarentena):,}")
print(f"Registros en Curated: {len(df_limpio):,}")
print(f"Total de registros conservados tras limpieza: {len(df_cuarentena) + len(df_limpio):,}")

Datos escritos en la Capa CURATED exitosamente.

--- ESTADÍSTICA DEL DATO CURADO ---
              monto monto_escalado monto_estandarizado
count  4,937,387.00   4,937,387.00        4,937,387.00
mean       1,252.50           0.50               -0.00
std          706.46           0.28                1.00
min            5.00           0.00               -1.77
25%          653.39           0.26               -0.85
50%        1,252.53           0.50                0.00
75%        1,851.88           0.74                0.85
max        2,500.00           1.00                1.77


,id_transaccion,categoria,estado,monto,monto_escalado,monto_estandarizado
101,101,Juguetes,Reembolsado,1015.580233,0.405042,-0.335363
102,102,Alimentos,Pendiente,1837.507104,0.734472,0.828080
103,103,Electronica,Completado,997.149883,0.397655,-0.361451
104,104,Ropa,Completado,2021.055379,0.808038,1.087894
105,105,Juguetes,Pendiente,1251.907721,0.499762,-0.000840



--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---


,id_transaccion,categoria,estado,monto,monto_escalado
101,101,Juguetes,Reembolsado,1015.580233,0.405042
102,102,Alimentos,Pendiente,1837.507104,0.734472
103,103,Electronica,Completado,997.149883,0.397655
104,104,Ropa,Completado,2021.055379,0.808038
105,105,Juguetes,Pendiente,1251.907721,0.499762



--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---
RAW (Sucio): 74.41 MB
QUARANTINE (Outliers): 0.00 MB
CURATED (Limpio): 151.75 MB

--- CONTROL DE REGISTROS ---
Registros en cuarentena: 101
Registros en Curated: 4,937,387
Total de registros conservados tras limpieza: 4,937,488
